# Template Matching
Find a logo/sticker in multiple product photos using TM_CCOEFF_NORMED

In [ ]:
import cv2
import argparse
import matplotlib.pyplot as plt


In [ ]:
image = cv2.imread("/content/coca-cola.png")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
plt.imshow(image)
plt.axis("off")

In [ ]:
template = cv2.imread("/content/template.png")
template = cv2.cvtColor(template, cv2.COLOR_BGR2RGB)
plt.imshow(template)
plt.axis("off")

In [ ]:
# Convert the main image from BGR (OpenCV default color format) to grayscale.
# Grayscale simplifies the image by removing color information and keeping only intensity values.
# This is useful for tasks like template matching because it reduces computation.
imageGray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Convert the template image from BGR to grayscale for the same reason.
# Both the source image and template must be in the same format for accurate comparison.
templateGray = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)

In [ ]:
result = cv2.matchTemplate(imageGray, templateGray, cv2.TM_CCOEFF_NORMED)  # Perform template matching to find where the template best fits in the grayscale image using normalized correlation

(minVal, maxVal, minLoc, maxLoc) = cv2.minMaxLoc(result)  # Get the minimum and maximum matching scores and their locations from the result matrix

In [ ]:
(startX, startY) = maxLoc  # Extract the top-left corner coordinates of the best template match location

endX = startX + template.shape[1]  # Calculate the bottom-right X coordinate by adding template width

endY = startY + template.shape[0]  # Calculate the bottom-right Y coordinate by adding template height

In [ ]:
cv2.rectangle(image, (startX, startY), (endX, endY), (0, 255, 0), 3)
# show the output image
plt.imshow(image)
plt.axis("off")


# Shape Matching

Classify simple shapes (circle,square,triangle) using Hu moments

In [ ]:
def get_hu_moments(image_path):
    img = cv2.imread(image_path)  # Read the image from the given file path
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # Convert the image to grayscale to simplify processing
    _, thresh = cv2.threshold(gray, 170, 255, 0)  # Apply thresholding to create a binary image separating object from background
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)  # Detect contours of objects in the binary image

    cnt = max(contours, key=cv2.contourArea)  # Select the largest contour assuming it is the main object
    moments = cv2.moments(cnt)  # Calculate spatial moments of the contour
    hu = cv2.HuMoments(moments)  # Compute the 7 Hu invariant moments from the contour moments
    return hu, cnt  # Return the Hu moments and the contour

In [ ]:
templates = {
    "Triangle": "/content/traingle.png",  # File path for the triangle template image
    "Square": "/content/square.png",      # File path for the square template image
    "Circle": "/content/circle.png"       # File path for the circle template image
}

template_contours = {}  # Dictionary to store contours extracted from template images

for name, path in templates.items():  # Iterate through each template name and its image path
    hu, cnt = get_hu_moments(path)  # Compute Hu moments and contour for the template image
    template_contours[name] = cnt  # Store the contour of the template using its shape name as the key

In [ ]:
input_path = '/content/shape_descriptor2.png'  # Path to the input image containing shapes to classify

img = cv2.imread(input_path)  # Read the input image
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # Convert the image to grayscale
_, thresh = cv2.threshold(gray, 170, 255, 0)  # Apply thresholding to create a binary image

contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)  # Detect contours of shapes in the image

for cnt in contours:  # Iterate through each detected contour (shape)
    best_match = None  # Store the name of the best matching template shape
    lowest_score = float("inf")  # Initialize lowest score with infinity (lower score = better match)

    for name, tmpl_cnt in template_contours.items():  # Compare current contour with each template contour
        score = cv2.matchShapes(cnt, tmpl_cnt, cv2.CONTOURS_MATCH_I1, 0)  # Compute similarity score between contours

        if score < lowest_score:  # Update best match if a lower score (better similarity) is found
            lowest_score = score
            best_match = name

    x, y = cnt[0, 0]  # Get a coordinate from the contour to place the label

    cv2.drawContours(img, [cnt], -1, (0, 255, 255), 3)  # Draw the detected contour on the image
    cv2.putText(img, best_match, (x, y),  # Write the predicted shape name near the contour
                cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                (255, 0, 0), 2)

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))  # Convert BGR to RGB for correct display in matplotlib
plt.axis('off')  # Hide axis for cleaner visualization

# Face detection: Run Haar cascade on webcam + group photos → count faces, draw boxes.


In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from google.colab import files

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'  # Load the pre-trained Haar Cascade model for frontal face detection
)

In [ ]:
uploaded = files.upload()

Saving group_photo.png to group_photo.png


In [ ]:
for filename in uploaded.keys():

    # Read image
    img = cv2.imread(filename)

    # Convert to grayscale (Haar works on gray images)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    print("Number of faces detected:", len(faces))

    # Draw bounding boxes
    for (x, y, w, h) in faces:
        cv2.rectangle(
            img,
            (x, y),
            (x+w, y+h),
            (0, 255, 0),
            2
        )

    # Show result
    cv2_imshow(img)

### Webcam Face Detection

In [ ]:
from IPython.display import Javascript
from google.colab.output import eval_js
from base64 import b64decode
import PIL.Image
import io

In [ ]:
def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getVideoTracks()[0].stop();
      div.remove();

      return canvas.toDataURL('image/jpeg', quality);
    }
  ''')

  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])

  with open(filename, 'wb') as f:
      f.write(binary)

  return filename

In [ ]:
photo = take_photo()
img = cv2.imread(photo)

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

faces = face_cascade.detectMultiScale(gray,1.3,5)

print("Faces detected:",len(faces))

for (x,y,w,h) in faces:
    cv2.rectangle(img,(x,y),(x+w,y+h),(255,0,0),2)

cv2_imshow(img)

# Build a “LEGO brick detector” : use template matching + contour filtering

In [48]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
from google.colab import files

In [ ]:
img_scene =  cv2.imread("/content/lego_scene2.png")
img_scene = cv2.cvtColor(img_scene, cv2.COLOR_BGR2RGB)
plt.imshow(img_scene)

In [ ]:
img_temp =  cv2.imread("/content/lego_template2.png")
img_temp = cv2.cvtColor(img_temp, cv2.COLOR_BGR2RGB)
plt.imshow(img_temp)

In [ ]:
img_scene_gray = cv2.cvtColor(img_scene, cv2.COLOR_RGB2GRAY)

# Apply adaptive thresholding to better separate objects with varying lighting
# You might need to adjust the block size and C parameter based on your image
img_scene_thresh = cv2.adaptiveThreshold(img_scene_gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)

# Find contours
contours, _ = cv2.findContours(img_scene_thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Create a copy of the original image to draw contours on
img_contours = img_scene.copy()

# Draw contours on the image
# Filter out very small contours that might be noise
lego_bricks = []
min_area = 100 # Adjust this value as needed

for i, contour in enumerate(contours):
    area = cv2.contourArea(contour)
    if area > min_area:
        # You can store the contours or draw them
        lego_bricks.append(contour)
        cv2.drawContours(img_contours, [contour], -1, (0, 255, 0), 2) # Green contours

print(f"Found {len(lego_bricks)} potential LEGO bricks.")

# Display the thresholded image and the image with contours
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img_scene_thresh, cmap='gray')
plt.title('Thresholded Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_contours)
plt.title('Detected LEGO Bricks (Contours)')
plt.axis('off')
plt.show()

In [ ]:
# Convert scene image to grayscale for template matching
img_scene_gray = cv2.cvtColor(img_scene, cv2.COLOR_RGB2GRAY)

# Convert template image to grayscale
img_temp_gray = cv2.cvtColor(img_temp, cv2.COLOR_RGB2GRAY)
(tH, tW) = img_temp_gray.shape[:2]

# Create a copy of the scene image to draw results on
img_result = img_scene.copy()

match_threshold = 0.53 # Adjusted threshold for matching sensitivity

# Define rotation angles to test (0, 90, 180, 270 degrees clockwise)
ROTATION_MODES = [
    ("0 deg", None),
    ("90 deg CW", cv2.ROTATE_90_CLOCKWISE),
    ("180 deg", cv2.ROTATE_180),
    ("270 deg CW", cv2.ROTATE_90_COUNTERCLOCKWISE)
]

all_detections = [] # Store (startX, startY, endX, endY, score, rotation_name)

print(f"Searching for all instances of the template (dim: {tW}x{tH}) using rotation-invariant matching.")

# Iterate through each rotation mode for the template
for rotation_name, rotation_mode in ROTATION_MODES:
    current_img_temp_rotated_gray = img_temp_gray
    current_tH, current_tW = tH, tW

    if rotation_mode is not None:
        current_img_temp_rotated = cv2.rotate(img_temp, rotation_mode)
        current_img_temp_rotated_gray = cv2.cvtColor(current_img_temp_rotated, cv2.COLOR_RGB2GRAY)
        current_tH, current_tW = current_img_temp_rotated_gray.shape[:2]

    # Perform template matching
    result = cv2.matchTemplate(img_scene_gray, current_img_temp_rotated_gray, cv2.TM_CCOEFF_NORMED)

    # Find all locations where the match score is above the threshold
    loc = np.where(result >= match_threshold)

    for pt in zip(*loc[::-1]): # Iterate over all found locations
        startX, startY = pt[0], pt[1]
        endX = startX + current_tW
        endY = startY + current_tH
        score = result[startY, startX]
        all_detections.append((startX, startY, endX, endY, score, rotation_name))

# Prepare data for NMS
bboxes = []
scores_for_nms = []
for det in all_detections:
    # Convert (startX, startY, endX, endY) to (x, y, w, h) for NMSBoxes
    x = det[0]
    y = det[1]
    w = det[2] - det[0]
    h = det[3] - det[1]
    bboxes.append([x, y, w, h])
    scores_for_nms.append(det[4]) # The score

# Apply non-maxima suppression to remove overlapping bounding boxes
# NMSBoxes returns the indices of the bounding boxes to keep
indices = cv2.dnn.NMSBoxes(bboxes, scores_for_nms, match_threshold, 0.3) # 0.3 is overlapThreshold (adjust as needed)

matched_count = 0
if len(indices) > 0:
    for i in indices.flatten():
        # Retrieve the original detection details from all_detections
        det_info = all_detections[i]
        startX, startY, endX, endY, score, rotation_name = det_info

        matched_count += 1

        cv2.rectangle(img_result, (startX, startY), (endX, endY), (0, 255, 0), 3) # Green rectangle
        cv2.putText(img_result, f"Match: {score:.2f} ({rotation_name})", (startX, startY - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

# Display the results with all matches highlighted
plt.figure(figsize=(12, 10))
plt.imshow(img_result)
plt.title(f"Rotation-Invariant Template Matching: {matched_count} matches found (threshold={match_threshold:.2f})")
plt.axis('off')
plt.show()

print(f"Total unique matched bricks found after NMS: {matched_count}")